# Step 35 — The matrix tests rerun on today's products: THS against RavKav 2022 and 2025, and the PCA

The diagnostics of steps 12–14 and 21 (cosine similarity and GEH, Kolmogorov–Smirnov on the
trip-length distribution, MSSIM, PCA subspace overlap) were built for the survey × cellular
hybrids. This notebook applies the same tests — same functions, same conventions — to what
changed on 23 September 2026: the **transit matrices rerun with the corrected mode codes**
(steps 15–16, §6m–§6n), and the **RavKav 2025 layer** of step 34 (§6af), against the THS
survey matrices and against the May 2022 RavKav products of steps 8–9. The car matrix did
not change today (no new car source; 1,353,798 trips in 2022) and enters only as the
reference structure in the PCA, as in step 21.

Matrices compared (all 06:00–09:00, both ends in the 778 study TAZs):

| label | product | unit | vintage |
|---|---|---|---|
| `THS bus 2018` | `two_mode/bus_survey_taz.csv` — survey Public Bus + Matronit, corrected codes | person journeys, residents | 2018 |
| `THS bus 2018 d1` / `d2` | the same by survey day (rebuilt here from the trips file) | | 2018 |
| `THS bus 2018 calibrated` | `two_mode/bus_calibrated_taz.csv` — step 15's segmented coverage rule | mixed | 2018 / 2022 |
| `THS bus 2022` | `three_mode_2022/bus_2022_taz.csv` — step 16 | | 2022 |
| `RavKav 2022 raw` | `bus/bus_od_taz_avg.csv` — linked journeys, RavKav's own inferred alightings | journeys, all riders | May 2022 |
| `RavKav 2022` | `bus/bus_od_taz_new.csv` — the same volumes on the OnBoard pattern (step 9) | | May 2022 |
| `RavKav 2025` | `ravkav_2025/bus_od_taz_2025.csv` — non-transfer boardings on the OnBoard pattern (step 34) | journey origins (tag-defined) | 2025 |
| `RavKav 2025 legs` | `ravkav_2025/bus_od_taz_2025_legs.csv` — all boardings on the OnBoard pattern | legs | 2025 |
| rail: `THS rail 2022`, `station 2019 × 0.793`, `rail 2025` | step 16, step 10, step 34 | door-to-door / station-to-station | |

Two frames sit inside every THS-vs-RavKav pair: residents' door-to-door journeys against all
riders' stop-to-stop journeys (§8 caveat 7), and — for 2025 — journey origins defined by the
transfer tag against linked journeys (caveat 16). The tests measure the agreement of the
products as they are; they do not remove the frame difference.

In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shapefile
from shapely.geometry import shape
from scipy.ndimage import uniform_filter
from scipy.linalg import subspace_angles
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
rng = np.random.default_rng(42)
BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/ths2017/tests'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)
def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)
def save_show(fig, name): fig.savefig(f'Output/figures/{name}', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()
def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')

## 1. Zones, orderings, distances, and the matrices

In [3]:
KEYS_LFS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
keys_raw = pd.read_csv('Input/taz_keys_from_shapefile.csv') if is_pointer(KEYS_LFS) else pd.read_csv(KEYS_LFS, encoding='windows-1255')
kd = keys_raw[['TAZ_1270', 'TAZ_NUMBER', 'SZ_NEW']].dropna(subset=['TAZ_NUMBER']).astype({'TAZ_NUMBER': int, 'SZ_NEW': int}).drop_duplicates('TAZ_NUMBER')
TAZ = np.array(sorted(kd['TAZ_NUMBER'])); N = len(TAZ); taz_pos = {t: i for i, t in enumerate(TAZ)}
sz_of = kd.set_index('TAZ_NUMBER')['SZ_NEW'].reindex(TAZ).values; SZ = sorted(set(sz_of)); sz_pos = {z: i for i, z in enumerate(SZ)}
M_sz = np.zeros((N, len(SZ)));
for i, z in enumerate(sz_of): M_sz[i, sz_pos[z]] = 1
sub = pd.read_excel('Input/Submatrix_tazs.xlsx'); AREA = sorted(sub['AggAreaCode'].unique()); area_of = sub.set_index('TAZ')['AggAreaCode']; area_names = sub.drop_duplicates('AggAreaCode').set_index('AggAreaCode')['AggAreaName']
M_area = np.zeros((N, len(AREA))); apos = {a: i for i, a in enumerate(AREA)}
for t, a in area_of.items():
    if t in taz_pos: M_area[taz_pos[t], apos[a]] = 1
xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); v2key = xl.parse('TazAgg').set_index('TAZ')['AggCode']; V2 = list(xl.parse('AreaCodes')['AggCode']); v2names = xl.parse('AreaCodes').set_index('AggCode')['AggAreaName']
M_v2 = np.zeros((N, len(V2))); vpos = {a: i for i, a in enumerate(V2)}
for t, a in v2key.items():
    if t in taz_pos: M_v2[taz_pos[t], vpos[a]] = 1
LEVELS = {'TAZ': None, 'SZ': M_sz, 'AREA': M_area, 'V2': M_v2}
def to_level(X, lvl): return X if LEVELS[lvl] is None else LEVELS[lvl].T @ X @ LEVELS[lvl]

sf = shapefile.Reader('Input/TAZ_North/TAZ_North.shp'); cent = {}
for sr in sf.shapeRecords(): cent[sr.record['TAZ_NUMBER']] = shape(sr.shape.__geo_interface__).centroid.coords[0]
xy = np.array([cent[t] for t in TAZ]); DIST = np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(axis=2)) / 1000.0
nn = np.where(np.eye(N, dtype=bool), np.inf, DIST).min(axis=1); np.fill_diagonal(DIST, nn / 2)
# orderings for the MSSIM windows: native (TAZ number), and superzone-grouped by centroid easting (the spatial proxy used here instead of step 14's Hilbert curve)
ORD = {'native': np.arange(N), 'sz-grouped': np.lexsort((xy[:, 0], sz_of))}

def load(path, index_all=True):
    m = pd.read_csv(path, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int)
    return m.reindex(index=TAZ, columns=TAZ, fill_value=0).fillna(0).values if index_all else m
SRC = {'THS bus 2018': load('Output/ths2017/two_mode/bus_survey_taz.csv'),
       'THS bus 2018 calibrated': load('Output/ths2017/two_mode/bus_calibrated_taz.csv'),
       'THS bus 2022': load('Output/ths2017/three_mode_2022/bus_2022_taz.csv'),
       'THS transit 2022': load('Output/final_2022/transit_2022_taz.csv'),
       'RavKav 2022 raw': load('Output/bus/bus_od_taz_avg.csv'),
       'RavKav 2022': load('Output/bus/bus_od_taz_new.csv'),
       'RavKav 2025': load('Output/ravkav_2025/bus_od_taz_2025.csv'),
       'RavKav 2025 legs': load('Output/ravkav_2025/bus_od_taz_2025_legs.csv'),
       'car 2022': load('Output/ths2017/three_mode_2022/car_2022_taz.csv')}

# survey bus by day, corrected codes, with the step-15 zone conversion (for the repeatability reference and the household bootstrap)
df = pd.read_excel('Input/THS_2017-2018/trips_ths_2017.xlsx')
d = df.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy(); g = d.groupby(['PerID3', 'SurveyDay'])
d['origin'] = g['actTaz'].shift(1); d['trip_dep_h'] = g['Dep_h'].shift(1)
trips = d.dropna(subset=['origin', 'actTaz', 'trip_dep_h']); trips = trips[trips['trip_dep_h'].isin([6, 7, 8])].copy()
BUS_CODES = [3, 5]                                                    # Public Bus, Matronit — the corrected codes (§6ae)
k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636'); map_2636_1250 = k26.set_index('TAZ_2636')['TAZ_1250']
trips['o1250'] = trips['origin'].map(map_2636_1250); trips['d1250'] = trips['actTaz'].map(map_2636_1250)
children = kd.groupby('TAZ_1270')['TAZ_NUMBER'].apply(list); north_1250 = sorted(children.index); z_idx = {z: i for i, z in enumerate(north_1250)}
tm = trips[trips['o1250'].isin(z_idx) & trips['d1250'].isin(z_idx)].copy(); tm['oi'] = tm['o1250'].map(z_idx); tm['di'] = tm['d1250'].map(z_idx)
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='windows-1255').set_index('TAZ_ID').reindex(TAZ); pop, emp = zon['POPULATION'].fillna(0).values, zon['EMPL_TOT'].fillna(0).values
def alloc(primary, secondary):
    S = np.zeros((len(north_1250), N))
    for z, kids in children.items():
        idx = [taz_pos[t] for t in kids]
        for v in (primary[idx], secondary[idx], np.ones(len(idx))):
            if v.sum() > 0: S[z_idx[z], idx] = v / v.sum(); break
    return S
S_o, S_d = alloc(pop, emp), alloc(emp, pop)
def m1250(sub_, w=None):
    M = np.zeros((len(north_1250),) * 2); np.add.at(M, (sub_['oi'].values, sub_['di'].values), sub_['new_wf'].values if w is None else w); return M
bus_tm = tm[tm['mainmode'].isin(BUS_CODES)]
for day in (1, 2): SRC[f'THS bus 2018 d{day}'] = S_o.T @ m1250(bus_tm[bus_tm['SurveyDay'] == day]) @ S_d
assert abs((SRC['THS bus 2018 d1'] + SRC['THS bus 2018 d2']).sum() / 2 - SRC['THS bus 2018'].sum()) < 1, 'day rebuild must reproduce the step-15 survey bus total'
print('totals (trips, 06:00–09:00, both ends in the study area):'); print(pd.Series({k: v.sum() for k, v in SRC.items()}).round(0).to_string())
PAIRS = [('THS bus 2018 d1', 'THS bus 2018 d2'), ('THS bus 2018', 'RavKav 2022'), ('THS bus 2018', 'RavKav 2022 raw'), ('THS bus 2022', 'RavKav 2022'), ('THS bus 2022', 'RavKav 2025'),
         ('THS bus 2018', 'RavKav 2025'), ('THS bus 2018 calibrated', 'RavKav 2025'), ('RavKav 2022', 'RavKav 2025'), ('RavKav 2022 raw', 'RavKav 2025'), ('RavKav 2025', 'RavKav 2025 legs'), ('THS transit 2022', 'RavKav 2025')]

totals (trips, 06:00–09:00, both ends in the study area):
THS bus 2018                125439.0
THS bus 2018 calibrated     127185.0
THS bus 2022                130779.0
THS transit 2022            134829.0
RavKav 2022 raw              92440.0
RavKav 2022                  92713.0
RavKav 2025                  99896.0
RavKav 2025 legs            103777.0
car 2022                   1353798.0
THS bus 2018 d1             127653.0
THS bus 2018 d2             123225.0


## 2. Cosine similarity and GEH (the step-12 tests)

Cosine on the raw cells and on the row-normalised profiles; GEH on hourly cell flows (three-hour
values ÷ 3), pass rates on the cells with a non-zero mean flow, flow-weighted. The survey's two
days are the repeatability reference: a pair that scores below the day-to-day agreement differs
by more than the survey's own sampling noise.

In [4]:
def cosine(a, b):
    a, b = np.asarray(a, float).ravel(), np.asarray(b, float).ravel(); na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na > 0 and nb > 0 else np.nan
def rownorm(X):
    rs = X.sum(axis=1, keepdims=True); return np.divide(X, rs, out=np.zeros_like(X, dtype=float), where=rs > 0)
def geh(m, c):
    m, c = np.asarray(m, float) / 3.0, np.asarray(c, float) / 3.0; s = m + c
    with np.errstate(divide='ignore', invalid='ignore'): return np.where(s > 0, np.sqrt(2 * (m - c) ** 2 / np.where(s == 0, 1, s)), np.nan)
def geh_summary(m, c):
    G = geh(m, c); w = (np.asarray(m, float) + np.asarray(c, float)) / 6.0; ok = ~np.isnan(G); G, w = G[ok], w[ok]
    return {'cells': int(ok.sum()), 'GEH<5': float((G < 5).mean()), 'GEH<5 (flow-wtd)': float(w[G < 5].sum() / w.sum()), 'GEH<10 (flow-wtd)': float(w[G < 10].sum() / w.sum()), 'median GEH': float(np.median(G))}
rows = []
for lvl in ['TAZ', 'SZ', 'AREA', 'V2']:
    L = {k: to_level(v, lvl) for k, v in SRC.items()}
    for a, b in PAIRS:
        A, B = L[a], L[b]; r = {'level': lvl, 'pair': f'{a} vs {b}', 'total A': A.sum(), 'total B': B.sum(), 'ratio B/A': B.sum() / A.sum(), 'cosine': cosine(A, B), 'cosine row-normalised': cosine(rownorm(A), rownorm(B)),
                                'cosine origins': cosine(A.sum(axis=1), B.sum(axis=1)), 'cosine destinations': cosine(A.sum(axis=0), B.sum(axis=0))}
        if lvl != 'TAZ': r.update(geh_summary(A, B))
        rows.append(r)
cg = pd.DataFrame(rows); cg.to_csv(f'{OUT}/ravkav2025_cosine_geh.csv', index=False, float_format='%.4f')
show = ['pair', 'ratio B/A', 'cosine', 'cosine row-normalised', 'cosine origins', 'cosine destinations', 'GEH<5 (flow-wtd)', 'median GEH']
for lvl in ['TAZ', 'SZ', 'AREA', 'V2']:
    print(f'\n=== {lvl}'); print(cg[cg['level'] == lvl][[c for c in show if c in cg]].round(3).to_string(index=False))


=== TAZ
                                  pair  ratio B/A  cosine  cosine row-normalised  cosine origins  cosine destinations  GEH<5 (flow-wtd)  median GEH
    THS bus 2018 d1 vs THS bus 2018 d2      0.965   0.667                  0.594           0.853                0.863               NaN         NaN
           THS bus 2018 vs RavKav 2022      0.739   0.059                  0.058           0.602                0.611               NaN         NaN
       THS bus 2018 vs RavKav 2022 raw      0.737   0.274                  0.160           0.600                0.639               NaN         NaN
           THS bus 2022 vs RavKav 2022      0.709   0.099                  0.137           0.628                0.600               NaN         NaN
           THS bus 2022 vs RavKav 2025      0.764   0.093                  0.135           0.581                0.590               NaN         NaN
           THS bus 2018 vs RavKav 2025      0.796   0.054                  0.055           0.550       

## 3. Kolmogorov–Smirnov on the trip-length distribution (the step-13 test)

Trip lengths are centroid distances (intra-TAZ cells at half the nearest-neighbour distance),
every cell weighted by its trips; D is the largest gap between two cumulative distributions,
reported with the distance at which it occurs. The survey's household bootstrap (200
replicates of the day-averaged bus matrix) gives the sampling band of D for the pairs that
involve the survey.

In [5]:
order = np.argsort(DIST, axis=None, kind='stable'); dist_sorted = DIST.ravel()[order]
def ecdf(X, mask=None):
    w = np.asarray(X, float).ravel()[order]
    if mask is not None: w = w * np.asarray(mask, bool).ravel()[order]
    tot = w.sum(); return np.cumsum(w) / tot if tot > 0 else np.full_like(w, np.nan)
def ks(FA, FB):
    gap = FA - FB; i = int(np.nanargmax(np.abs(gap))); return float(abs(gap[i])), float(dist_sorted[i]), float(gap[i])
def wquantile(X, q, mask=None): F = ecdf(X, mask); return float(dist_sorted[np.searchsorted(F, q)])
def tld_stats(X, mask=None):
    w = np.asarray(X, float) * (1 if mask is None else mask)
    return {'trips': w.sum(), 'mean km': float((w * DIST).sum() / w.sum()), 'median km': wquantile(X, .5, mask), 'p90 km': wquantile(X, .9, mask), 'share < 1 km': float(w[DIST < 1].sum() / w.sum()), 'share < 3 km': float(w[DIST < 3].sum() / w.sum())}
HH = bus_tm['HHID3'].values; hh_ids, hh_inv = np.unique(HH, return_inverse=True); oi, di, wf, day = bus_tm['oi'].values, bus_tm['di'].values, bus_tm['new_wf'].values, bus_tm['SurveyDay'].values
def boot_matrix():
    cnt = np.bincount(rng.integers(0, len(hh_ids), len(hh_ids)), minlength=len(hh_ids)); w = wf * cnt[hh_inv]
    M1, M2 = np.zeros((len(north_1250),) * 2), np.zeros((len(north_1250),) * 2)
    np.add.at(M1, (oi[day == 1], di[day == 1]), w[day == 1]); np.add.at(M2, (oi[day == 2], di[day == 2]), w[day == 2])
    return S_o.T @ ((M1 + M2) / 2) @ S_d
BOOT = [boot_matrix() for _ in range(200)]
MASKS = {'all cells': None, 'excl. intra-TAZ': ~np.eye(N, dtype=bool)}
KS_SRC = ['THS bus 2018', 'THS bus 2018 d1', 'THS bus 2018 d2', 'THS bus 2022', 'RavKav 2022 raw', 'RavKav 2022', 'RavKav 2025', 'RavKav 2025 legs', 'car 2022']
rows = []
for mname, mask in MASKS.items():
    F = {k: ecdf(SRC[k], mask) for k in KS_SRC}; FB = [ecdf(b, mask) for b in BOOT]
    for a, b in PAIRS + [('THS bus 2018', 'car 2022'), ('RavKav 2025', 'car 2022')]:
        if a not in F or b not in F: continue
        D, x, sg = ks(F[a], F[b]); r = {'cells': mname, 'pair': f'{a} vs {b}', 'D': D, 'at km': x, 'signed gap (A−B)': sg}
        if 'THS bus 2018' in (a, b):
            other = F[b] if a == 'THS bus 2018' else F[a]; Db = np.array([ks(fb, other)[0] for fb in FB]); r['D boot p2.5'], r['D boot p97.5'] = np.percentile(Db, 2.5), np.percentile(Db, 97.5)
        rows.append(r)
kst = pd.DataFrame(rows); kst.to_csv(f'{OUT}/ravkav2025_ks_tld.csv', index=False, float_format='%.4f')
stats = pd.DataFrame({k: tld_stats(SRC[k]) for k in KS_SRC}).T; stats.to_csv(f'{OUT}/ravkav2025_tld_stats.csv', float_format='%.4f')
print('trip-length statistics (centroid km):'); print(stats.round(3).to_string()); print(); print(kst.round(3).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), facecolor='white', sharey=True)
for ax, (mname, mask) in zip(axes, MASKS.items()):
    for name, color in [('THS bus 2018', BLUE), ('THS bus 2022', INK2), ('RavKav 2022', AQUA), ('RavKav 2025', ORANGE), ('car 2022', MUTED)]:
        ax.plot(dist_sorted, ecdf(SRC[name], mask), color=color, linewidth=1.6, label=name)
    ax.set_xscale('symlog', linthresh=1); ax.set_xlim(0, 120); style_ax(ax, f'Trip-length distribution — {mname}', 'centroid distance (km, symlog)', 'cumulative share of trips' if ax is axes[0] else None); ax.legend(frameon=False, fontsize=8, loc='upper left')
save_show(fig, 'ravkav2025_ks_tld.png')

trip-length statistics (centroid km):
                        trips  mean km  median km  p90 km  share < 1 km  share < 3 km
THS bus 2018       125438.752    6.715      3.227  17.268         0.124         0.480
THS bus 2018 d1    127652.652    6.964      3.330  17.724         0.115         0.468
THS bus 2018 d2    123224.852    6.457      3.105  16.155         0.133         0.492
THS bus 2022       130778.549    8.520      4.908  20.638         0.096         0.336
RavKav 2022 raw     92439.750    7.081      3.577  19.157         0.123         0.439
RavKav 2022         92712.819   13.817      6.390  37.687         0.072         0.292
RavKav 2025         99896.273   13.966      6.410  38.123         0.069         0.283
RavKav 2025 legs   103777.310   13.803      6.345  37.687         0.069         0.284
car 2022          1353797.747    6.014      2.186  16.733         0.233         0.574

          cells                               pair     D  at km  signed gap (A−B)  D boot p2.5  D boo

## 4. MSSIM (the step-14 test)

Mean structural similarity over sliding windows on the matrix as an image, luminance ×
contrast × structure, on the raw and the log-transformed cells; at TAZ level in the
superzone-grouped order (windows 5 and 9), at superzone and area level in the native order
(windows 3 and 5 / 3). The chance level is the same statistic with one matrix's zones permuted.

In [6]:
def ssim_terms(A, B, w, L=None):
    A, B = np.asarray(A, float), np.asarray(B, float); L = max(A.max(), B.max()) if L is None else L; C1, C2 = (0.01 * L) ** 2, (0.03 * L) ** 2
    f = lambda X: uniform_filter(X, size=w, mode='reflect'); mu_a, mu_b = f(A), f(B)
    va = np.maximum(f(A * A) - mu_a ** 2, 0); vb = np.maximum(f(B * B) - mu_b ** 2, 0); cov = f(A * B) - mu_a * mu_b; sa, sb = np.sqrt(va), np.sqrt(vb)
    lum = (2 * mu_a * mu_b + C1) / (mu_a ** 2 + mu_b ** 2 + C1); con = (2 * sa * sb + C2) / (va + vb + C2); stru = (cov + C2 / 2) / (sa * sb + C2 / 2)
    h = w // 2; sl = slice(h, A.shape[0] - h) if h > 0 else slice(None)
    return {'lum': lum[sl, sl], 'con': con[sl, sl], 'str': stru[sl, sl], 'ssim': (lum * con * stru)[sl, sl]}
def reorder(X, o): return X[np.ix_(o, o)]
def mssim(A, B, w, o=None, transform=None):
    if o is not None: A, B = reorder(A, o), reorder(B, o)
    if transform == 'log': A, B = np.log1p(A), np.log1p(B)
    t = ssim_terms(A, B, w); return {'MSSIM': float(t['ssim'].mean()), 'luminance': float(t['lum'].mean()), 'contrast': float(t['con'].mean()), 'structure': float(t['str'].mean())}
MS_PAIRS = [('THS bus 2018 d1', 'THS bus 2018 d2'), ('THS bus 2018', 'RavKav 2022'), ('THS bus 2022', 'RavKav 2025'), ('RavKav 2022', 'RavKav 2025'), ('THS bus 2018', 'RavKav 2025'), ('RavKav 2022 raw', 'RavKav 2025')]
rows = []
for lvl, windows, o in [('TAZ', [5, 9], ORD['sz-grouped']), ('SZ', [3, 5], None), ('AREA', [3], None)]:
    L = {k: to_level(SRC[k], lvl) for k in SRC}
    for w in windows:
        for tr in ['raw', 'log']:
            for a, b in MS_PAIRS:
                r = {'level': lvl, 'window': w, 'scale': tr, 'pair': f'{a} vs {b}', **mssim(L[a], L[b], w, o, tr)}
                nul = [mssim(L[a] if o is None else reorder(L[a], o), reorder(reorder(L[b], rng.permutation(L[b].shape[0])), o if o is not None else np.arange(L[b].shape[0])), w, None, tr)['MSSIM'] for _ in range(20 if lvl == 'TAZ' else 40)]
                r['null mean (B permuted)'] = float(np.mean(nul)); r['null p95'] = float(np.percentile(nul, 95)); rows.append(r)
ms = pd.DataFrame(rows); ms.to_csv(f'{OUT}/ravkav2025_mssim.csv', index=False, float_format='%.4f')
print(ms[(ms['scale'] == 'log')].round(3).to_string(index=False))

level  window scale                               pair  MSSIM  luminance  contrast  structure  null mean (B permuted)  null p95
  TAZ       5   log THS bus 2018 d1 vs THS bus 2018 d2  0.958      0.972     0.972      0.988                   0.814     0.817
  TAZ       5   log        THS bus 2018 vs RavKav 2022  0.833      0.894     0.882      0.960                   0.740     0.743
  TAZ       5   log        THS bus 2022 vs RavKav 2025  0.760      0.845     0.871      0.918                   0.641     0.645
  TAZ       5   log         RavKav 2022 vs RavKav 2025  0.989      0.995     0.994      0.999                   0.683     0.686
  TAZ       5   log        THS bus 2018 vs RavKav 2025  0.831      0.893     0.881      0.960                   0.739     0.741
  TAZ       5   log     RavKav 2022 raw vs RavKav 2025  0.777      0.884     0.878      0.919                   0.631     0.635
  TAZ       9   log THS bus 2018 d1 vs THS bus 2018 d2  0.933      0.968     0.960      0.977           

## 5. PCA — destination structure: car, THS bus, RavKav 2022 and 2025 (the step-21 test)

Row-normalised destination profiles per origin at superzone and sub-area level; the principal
subspace of each matrix (k = the components holding 80 % of the car matrix's variance, as in
step 21); the overlap between two subspaces as the mean cos² of the principal angles, judged
against the survey's day-to-day repeatability and a permuted-geography null.

In [7]:
def pca(P):
    X = np.asarray(P, float); Xc = X - X.mean(axis=0); U, s, Vt = np.linalg.svd(Xc, full_matrices=False); lam = s ** 2 / max(X.shape[0] - 1, 1)
    return {'V': Vt.T, 'lam': lam, 'evr': lam / lam.sum(), 'scores': U * s}
def n_comp(evr, thr): return int(np.searchsorted(np.cumsum(evr), thr) + 1)
def mean_cos2(Va, Vb, k): return float(np.mean(np.cos(subspace_angles(Va[:, :k], Vb[:, :k])) ** 2))
def perm_null(Va, Vb, k, n_perm, seed=0):
    r = np.random.default_rng(seed); p = Va.shape[0]; return np.array([mean_cos2(Va[r.permutation(p)], Vb, k) for _ in range(n_perm)])
PCA_SRC = ['car 2022', 'THS bus 2018', 'THS bus 2018 d1', 'THS bus 2018 d2', 'THS bus 2022', 'RavKav 2022 raw', 'RavKav 2022', 'RavKav 2025']
PCA_PAIRS = [('car 2022', 'THS bus 2022'), ('car 2022', 'RavKav 2025'), ('THS bus 2018 d1', 'THS bus 2018 d2'), ('THS bus 2018', 'RavKav 2022'), ('THS bus 2022', 'RavKav 2025'), ('RavKav 2022', 'RavKav 2025'), ('RavKav 2022 raw', 'RavKav 2025'), ('THS bus 2018', 'RavKav 2022 raw')]
rows = []; PCS = {}
for lvl in ['SZ', 'AREA', 'V2']:
    L = {k: to_level(SRC[k], lvl) for k in PCA_SRC}
    ok = np.ones(L['car 2022'].shape[0], bool)
    for k in PCA_SRC: ok &= L[k].sum(axis=1) > 0
    P = {k: rownorm(L[k])[ok] for k in PCA_SRC}; PC = {k: pca(v) for k, v in P.items()}; PCS[lvl] = (P, PC, ok)
    k = min(n_comp(PC['car 2022']['evr'], .8), int(ok.sum()) - 2)
    for a, b in PCA_PAIRS:
        obs = mean_cos2(PC[a]['V'], PC[b]['V'], k); null = perm_null(PC[a]['V'], PC[b]['V'], k, 500, seed=1)
        rows.append({'level': lvl, 'origins': int(ok.sum()), 'k': k, 'pair': f'{a} vs {b}', 'overlap': obs, 'null mean': null.mean(), 'null p95': np.percentile(null, 95), 'permutation p': (1 + (null >= obs).sum()) / 501,
                     'mean profile distance': float(np.linalg.norm(P[a] - P[b], axis=1).mean())})
pc = pd.DataFrame(rows); pc.to_csv(f'{OUT}/ravkav2025_pca_overlap.csv', index=False, float_format='%.4f'); print(pc.round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(11, 5), facecolor='white')
sz = pc[pc['level'] == 'SZ']; y = np.arange(len(sz))
ax.barh(y, sz['overlap'], height=0.55, color=[ORANGE if 'd1' in p else BLUE for p in sz['pair']]); ax.scatter(sz['null p95'], y, marker='|', s=200, color=INK, label='permuted-geography null, p95', zorder=3)
ax.set_yticks(y); ax.set_yticklabels(sz['pair'], fontsize=9); ax.invert_yaxis(); ax.set_xlim(0, 1); ax.legend(frameon=False, fontsize=9, loc='lower right')
style_ax(ax, f"Superzone destination structure: subspace overlap (k = {int(sz['k'].iloc[0])}, {int(sz['origins'].iloc[0])} origins); orange = the survey's day-to-day reference", 'mean cos² of principal angles', None)
save_show(fig, 'ravkav2025_pca_overlap.png')

level  origins  k                               pair  overlap  null mean  null p95  permutation p  mean profile distance
   SZ       35 20           car 2022 vs THS bus 2022    0.743      0.574     0.641          0.002                  0.326
   SZ       35 20            car 2022 vs RavKav 2025    0.577      0.572     0.632          0.445                  0.444
   SZ       35 20 THS bus 2018 d1 vs THS bus 2018 d2    0.828      0.574     0.644          0.002                  0.208
   SZ       35 20        THS bus 2018 vs RavKav 2022    0.616      0.571     0.637          0.134                  0.434
   SZ       35 20        THS bus 2022 vs RavKav 2025    0.598      0.571     0.637          0.267                  0.408
   SZ       35 20         RavKav 2022 vs RavKav 2025    0.978      0.571     0.633          0.002                  0.043
   SZ       35 20     RavKav 2022 raw vs RavKav 2025    0.653      0.572     0.645          0.030                  0.306
   SZ       35 20    THS bus 201

## 6. Rail: survey, 2019 stations, 2025 stations

In [8]:
rs = pd.read_csv('Output/ths2017/three_mode_2022/rail_2022_taz.csv', index_col=0); rs.index = rs.index.astype(int); rs.columns = rs.columns.astype(int)
r19 = pd.read_csv('Output/ths2017/three_mode_2022/rail_station_smartcard_2022_taz.csv', index_col=0); r19.index = r19.index.astype(int); r19.columns = r19.columns.astype(int)
r25 = pd.read_csv('Output/ravkav_2025/rail_od_taz_2025.csv', index_col=0); r25.index = r25.index.astype(int); r25.columns = r25.columns.astype(int)
def to_sz(m):
    long = m.stack().reset_index(); long.columns = ['o', 'd', 'v']; long['O'] = long['o'].map(dict(zip(TAZ, sz_of))); long['D'] = long['d'].map(dict(zip(TAZ, sz_of)))
    return long.dropna().groupby(['O', 'D'])['v'].sum().unstack().reindex(index=SZ, columns=SZ, fill_value=0).fillna(0)
common = sorted(set(r19.index) & set(r25.index)); a19, a25 = r19.loc[common, common].values, r25.loc[common, common].values
rail = pd.DataFrame([
    {'comparison': 'station 2019 × 0.793 vs rail 2025, common station TAZs', 'n': len(common), 'total A': a19.sum(), 'total B': a25.sum(), 'cosine': cosine(a19, a25), 'cosine row-normalised': cosine(rownorm(a19), rownorm(a25)), **geh_summary(a19, a25)},
    {'comparison': 'THS rail 2022 (door-to-door) vs rail 2025 (station), superzones', 'n': len(SZ), 'total A': rs.values.sum(), 'total B': r25.values.sum(), 'cosine': cosine(to_sz(rs).values, to_sz(r25).values), 'cosine row-normalised': cosine(rownorm(to_sz(rs).values), rownorm(to_sz(r25).values)), **geh_summary(to_sz(rs).values, to_sz(r25).values)},
    {'comparison': 'THS rail 2022 vs station 2019 × 0.793, superzones', 'n': len(SZ), 'total A': rs.values.sum(), 'total B': r19.values.sum(), 'cosine': cosine(to_sz(rs).values, to_sz(r19).values), 'cosine row-normalised': cosine(rownorm(to_sz(rs).values), rownorm(to_sz(r19).values)), **geh_summary(to_sz(rs).values, to_sz(r19).values)}])
rail.to_csv(f'{OUT}/ravkav2025_rail_tests.csv', index=False, float_format='%.4f'); print(rail.round(3).to_string(index=False))
orig = pd.DataFrame({'THS rail 2022 origins': to_sz(rs).sum(axis=1), 'station 2019 × 0.793 origins': to_sz(r19).sum(axis=1), 'rail 2025 origins': to_sz(r25).sum(axis=1)}); orig = orig[orig.sum(axis=1) > 0]
print('rail origins by superzone:'); print(orig.round(0).to_string())

                                                     comparison  n  total A  total B  cosine  cosine row-normalised  cells  GEH<5  GEH<5 (flow-wtd)  GEH<10 (flow-wtd)  median GEH
         station 2019 × 0.793 vs rail 2025, common station TAZs 14 1799.472 3651.593   0.972                  0.966    175  0.971             0.705              1.000       0.631
THS rail 2022 (door-to-door) vs rail 2025 (station), superzones 36 4050.251 9437.106   0.582                  0.256    200  0.720             0.237              0.675       2.520
              THS rail 2022 vs station 2019 × 0.793, superzones 36 4050.251 4387.893   0.389                  0.203    173  0.815             0.312              0.757       1.781
rail origins by superzone:
    THS rail 2022 origins  station 2019 × 0.793 origins  rail 2025 origins
O                                                                         
1                   567.0                           0.0             1951.0
2                   819.0       

## 7. Summary table

In [9]:
summ = []
for lvl in ['SZ', 'AREA']:
    c_ = cg[cg['level'] == lvl].set_index('pair'); m_ = ms[(ms['level'] == lvl) & (ms['scale'] == 'log') & (ms['window'] == 3)].set_index('pair'); p_ = pc[pc['level'] == lvl].set_index('pair'); k_ = kst[kst['cells'] == 'all cells'].set_index('pair')
    for pair in ['THS bus 2018 d1 vs THS bus 2018 d2', 'THS bus 2018 vs RavKav 2022', 'THS bus 2022 vs RavKav 2025', 'RavKav 2022 vs RavKav 2025', 'THS bus 2018 vs RavKav 2025', 'RavKav 2022 raw vs RavKav 2025']:
        summ.append({'level': lvl, 'pair': pair, 'ratio B/A': c_.loc[pair, 'ratio B/A'], 'cosine': c_.loc[pair, 'cosine'], 'cosine row-norm.': c_.loc[pair, 'cosine row-normalised'], 'GEH<5 (flow-wtd)': c_.loc[pair, 'GEH<5 (flow-wtd)'],
                     'KS D (TAZ, all cells)': k_.loc[pair, 'D'] if pair in k_.index else np.nan, 'MSSIM (log, w=3)': m_.loc[pair, 'MSSIM'] if pair in m_.index else np.nan, 'PCA overlap': p_.loc[pair, 'overlap'] if pair in p_.index else np.nan})
summary = pd.DataFrame(summ); summary.to_csv(f'{OUT}/ravkav2025_tests_summary.csv', index=False, float_format='%.4f'); print(summary.round(3).to_string(index=False))

level                               pair  ratio B/A  cosine  cosine row-norm.  GEH<5 (flow-wtd)  KS D (TAZ, all cells)  MSSIM (log, w=3)  PCA overlap
   SZ THS bus 2018 d1 vs THS bus 2018 d2      0.965   0.888             0.911             0.461                  0.038             0.766        0.828
   SZ        THS bus 2018 vs RavKav 2022      0.739   0.623             0.618             0.207                  0.211             0.223        0.616
   SZ        THS bus 2022 vs RavKav 2025      0.764   0.591             0.646             0.287                  0.177             0.444        0.598
   SZ         RavKav 2022 vs RavKav 2025      1.077   0.981             0.993             0.911                  0.010             0.971        0.978
   SZ        THS bus 2018 vs RavKav 2025      0.796   0.628             0.615             0.225                  0.213             0.219          NaN
   SZ     RavKav 2022 raw vs RavKav 2025      1.081   0.725             0.730             0.504     